# Deliverable 4 – INF201

**Group members:** _[Mathias Gundersen and Jakub Mermer]_
**GitHub repository:** _[https://github.com/saltonese/Del_4-G6_5.git]_

This notebook contains

1. **Task 1 – Temperatures**: the `Temperature` class, a demonstration, and the `pytest` output
2. **Task 2 – Fractions**: the `Fraction` class, a demonstration, and the `pytest` output
3. **Discussion of GenAI use**

The classes live in `Task_1_temperature.py` and `Task_2_fraction.py`, and the tests in `test_temperature.py` and
`test_fraction.py`. The notebook imports the classes from those files and runs the tests with `pytest`.

In [1]:
import sys
import pytest
from pathlib import Path
from IPython.display import Code

project_dir = Path.cwd()
if not (project_dir / "Scripts").is_dir():
    project_dir = project_dir.parent

sys.path.insert(0, str(project_dir / "Scripts"))

print("Python:", sys.version.split()[0])
print("pytest:", pytest.__version__)
print(Path.cwd())


Python: 3.13.14
pytest: 9.2.0
c:\Users\mathi\StudieNMBU\PROG\Del_4-G6_5\Notebooks


---
# Task 1: Temperatures

## Design choices BESKRIV valgene du tar


- **Case-insensitive units.** The unit is normalised to upper case, so `"c"` and `"C"` give the same object.
- **Absolute zero.** Becaus absolute zero values for Fahrenheit and Celsius, when converted to Kelvin can become negative, we use `"1e-9"` as tolerance instead of 0 to avoid acidental ValueError in __init__.
- **


## The code

In [ ]:
class Temperature:
    def __init__(self, value, unit="C"):
        if unit.upper() not in ("C", "F", "K"):
            raise ValueError("Invalid unit. Use C, F or K.")
        self._value = float(value)
        self._unit = unit.upper()
        if self.to_kelvin() < -1e-9:
            raise ValueError("Temperature cannot be below absolute zero.")

    @property
    def value(self):
        return self._value

    @property
    def unit(self):
        return self._unit
    
    def to_celsius(self) -> float:
        if self.unit.upper() == 'C':
            return self.value
        if self.unit.upper() == 'F':
            return (self.value - 32) * (5 / 9)
        if self.unit.upper() == 'K':
            return self.value - 273.15
        
    def to_fahrenheit(self) -> float:
        if self.unit.upper() == 'C':
            return (self.value * (9 / 5)) + 32
        if self.unit.upper() == 'F':
            return self.value
        if self.unit.upper() == 'K':
            return (self.value - 273.15) * (9 / 5) + 32

    def to_kelvin(self) -> float:
        if self.unit.upper() == 'C':
            return self.value + 273.15
        if self.unit.upper() == 'F':
            return (self.value - 32) * (5 / 9) + 273.15
        if self.unit.upper() == 'K':
            return self.value

    def convert_to(self, target_unit: str):
        if target_unit.upper() == 'C':
            value = self.to_celsius()
        elif target_unit.upper() == 'F':
            value = self.to_fahrenheit()
        elif target_unit.upper() == 'K':
            value = self.to_kelvin()
        else:
            raise ValueError("Invalid unit. Please use 'C', 'F', or 'K'.")
        return Temperature(value, target_unit)

    def __repr__(self) -> str:
        return f"Temperature({self.value}, '{self.unit}')"

    def __str__(self) -> str:
        if self.unit.upper() in ("C", "F"):
            return f"{self.value:.2f} °{self.unit.upper()}"
        elif self.unit.upper() == "K":
            return f"{self.value:.2f} {self.unit.upper()}"
            
    def __eq__(self, other):
        if isinstance(other, Temperature):
            temp1 = self.to_celsius()
            temp2 = other.to_celsius()
            if abs(temp1 - temp2) < 1e-9:
                return True
            else:
                return False
        return NotImplemented

    def __ne__(self, other):
        if isinstance(other, Temperature):
            temp1 = self.to_celsius()
            temp2 = other.to_celsius()
            if abs(temp1 - temp2) >= 1e-9:
                return True
            else:
                return False
        return NotImplemented

    def __lt__(self, other):
        if isinstance(other, Temperature):
            temp1 = self.to_celsius()
            temp2 = other.to_celsius()
            if temp1 - temp2 < -1e-9:
                return True
            else:
                return False
        return NotImplemented
                
    def __le__(self, other):
        if isinstance(other, Temperature):
            return self.__lt__(other) or self.__eq__(other)
        return NotImplemented
    
    def __gt__(self, other):
        if isinstance(other, Temperature):
            temp1 = self.to_celsius()
            temp2 = other.to_celsius()
            if temp1 - temp2 > 1e-9:
                return True
            else:
                return False
        return NotImplemented
    
    def __ge__(self, other):
        if isinstance(other, Temperature):
            return self.__gt__(other) or self.__eq__(other)
        return NotImplemented
        

## Demonstraion

In [ ]:
from task_1_temperature import Temperature
temp1 = Temperature(0)
temp2 = Temperature(32, "F")

print(f"temp1: {temp1}, temp2: {temp2}")

temp3 = Temperature(100, "C")
temp4 = Temperature(230.15, "F")

temp3.__repr__()
temp4.__str__()

print(temp2.to_kelvin())
print(temp3.to_fahrenheit())
print(temp4.to_celsius())
print()
print(temp3 == temp4)
print(temp3 != temp4)
print(temp3 < temp4)
print(temp3 <= temp4)
print(temp3 > temp4)
print(temp3 >= temp4)


: 

- **Testing kode with pytest.**
By importing pytest we can check if different aspects of the main code of `"Task_1"` is functioning corectly, as intended.

---
# Task 2: Fractions

## Design choices

- **Validation.** Numerator and denominator must be integers (`TypeError` otherwise; `bool` is rejected explicitly,
  because it is a subclass of `int` in Python) and the denominator must be strictly positive (`ValueError`).
- **Lowest terms.** Fractions are reduced with `math.gcd` in the constructor, so `Fraction(2, 4)` is stored as `1/2`.
  This keeps equality and hashing simple.
- **Comparisons use cross-multiplication**, with no floating point: `a/b < c/d` is the same as `a*d < c*b`.
  This is only correct when both denominators are positive (multiplying an inequality by a negative number flips it),
  which is exactly why the task requires `denominator > 0`. The sign of a fraction is therefore always kept in the numerator.
- **Special methods implemented:** `__init__`, all six comparisons, and in addition `__repr__`, `__str__`, `__add__`,
  `__sub__`, `__mul__`, `__truediv__`, `__neg__`, `__abs__`, `__float__` and `__hash__` (the task requires at least four).
- **Operations with other types** return `NotImplemented`, so Python raises `TypeError`.
- **Hashable.** Unlike `Temperature`, this is safe, because equality is exact and fractions are in lowest terms.
  Equal fractions therefore always have the same `(numerator, denominator)` and the same hash.

## The code

In [ ]:

Code(filename=r"..\Scripts\Task_2_fraction.py", language="python")

## Demonstration

In [ ]:
from Task_2_fraction import Fraction

a, b = Fraction(1, 2), Fraction(1, 3)
print("repr:", repr(a), "| str:", a)
print("Reduced to lowest terms:", repr(Fraction(6, 8)))
print("Whole number prints without denominator:", Fraction(4, 2))
print()

print("Arithmetic with a =", a, "and b =", b)
print("  a + b =", a + b)
print("  a - b =", a - b)
print("  a * b =", a * b)
print("  a / b =", a / b)
print("  -a    =", -a, "| abs(-3/4) =", abs(Fraction(-3, 4)))
print("  float(a) =", float(a))
print("  1/2 / (-1/4) =", Fraction(1, 2) / Fraction(-1, 4), "(sign stays in the numerator)")

In [ ]:
print("Comparisons with a =", a, "and b =", b)
print("  a == b:", a == b, "| a == 2/4:", a == Fraction(2, 4))
print("  a != b:", a != b)
print("  a <  b:", a < b, "| a <= b:", a <= b)
print("  a >  b:", a > b, "| a >= b:", a >= b)
print()

print("Sorting:", [str(f) for f in sorted([Fraction(3, 4), Fraction(-1, 2), Fraction(1, 3), Fraction(1, 2)])])
print("Equal fractions in a set:", {Fraction(1, 2), Fraction(2, 4), Fraction(1, 3)})

In [ ]:
print("Invalid input and operations:")
for args in [(1, 0), (1, -2)]:
    try:
        Fraction(*args)
    except ValueError as error:
        print(f"  Fraction{args} -> ValueError: {error}")
for args in [(1.5, 2), ("1", 2)]:
    try:
        Fraction(*args)
    except TypeError as error:
        print(f"  Fraction{args} -> TypeError: {error}")
try:
    Fraction(1, 2) / Fraction(0, 5)
except ZeroDivisionError as error:
    print("  1/2 / 0 -> ZeroDivisionError:", error)
try:
    Fraction(1, 2) + 1
except TypeError as error:
    print("  Fraction(1, 2) + 1 -> TypeError:", error)

## Tests

`test_fraction.py` is organised as follows:

| Test class | What it checks |
|---|---|
| `TestInitialiser` | stored values, default denominator, reduction, negative and zero numerators, invalid denominators (`ValueError`), non-integers (`TypeError`), read-only attributes |
| `TestComparisons` | `==`, `!=`, `<`, `<=`, `>`, `>=` including negative fractions and different denominators, `NotImplemented` for other types, sorting |
| `TestStringRepresentations` | `__repr__` (including `eval(repr(f)) == f`) and `__str__` |
| `TestArithmetic` | `+`, `-`, `*`, `/` (also division by a negative fraction and by zero), `-f`, `abs(f)`, no mutation of the operands, `TypeError` for other types |
| `TestConversionAndHashing` | `float(f)`, equal hashes for equal fractions, use in sets and as dictionary keys |

In [ ]:
!{sys.executable} -m pytest "..\Scripts\test_fraction.py" -v --no-header -p no:cacheprovider

## All test togehter

## Use of generative AI
We used Claude (Anthropic, model: Claude Sonnet 5.5) to help and guide us though ths assignment. Here is a summary of the conversation.

### What the AI did

1. **Getting started.**  We uploaded the delvierable assaginment text and asked claude to point us in the right direction to start out. Claude explained the objective of the assignmet and helped us understand how to use stirng representation and pytest.raises. Claude also helped us getting used to using pytest from the terminal. Finally to start us off Claude made a jupyter notebook with no code, only text hints in markdown ready for us to use.
  
2. **Testing.**  Claude helped us verify that our classes and functions behaved like we wanted them to. (ran a quick test before setting up the test script)

3. **Video.** Claude made recomendations for important points to cover in the video.
  